## 1. Objetivo del notebook

El objetivo de este notebook es realizar una primera exploración del dataset integrado a nivel jugador-temporada (`player_season_features`), con el fin de:

- Verificar la correcta construcción del pipeline de datos (ingestión, validación, integración y feature engineering)
- Analizar la estructura del dataset resultante
- Identificar posibles problemas de calidad de datos (nulos, duplicados, inconsistencias)
- Comprender la distribución inicial de las variables clave, especialmente el valor de mercado y las métricas de rendimiento

Este análisis se sitúa dentro de la fase de *Data Understanding* del proceso CRISP-DM y sirve como base para el posterior modelado del valor de mercado.

## 2. Carga del dataset integrado

In [1]:
from pathlib import Path
import pandas as pd

# Detectar raíz del proyecto
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "data" / "processed" / "player_season_features.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset cargado correctamente")

Dataset cargado correctamente


# 3. Dimensiones del dataset

In [2]:
n_rows, n_cols = df.shape

pd.DataFrame({
    "Metric": ["Filas", "Columnas"],
    "Valor": [n_rows, n_cols]
})

,Metric,Valor
0,Filas,3
1,Columnas,40


## 4. Vista inicial de registros

In [3]:
df.head()

,player_name_tm,season,age,position_tm,club,league_tm,market_value_eur,player_id,season_start_year,position_group,...,z_progressive_passes_per90,z_progressive_carries_per90,z_tackles_per90,z_interceptions_per90,finishing_index,playmaking_index,progression_index,defensive_index,minutes_bucket,is_low_minutes
0,Antonio Silva,2022-2023,19,Centre-Back,Benfica,Liga Portugal,25000000,575677ef98189b21f805574af1733c49,2022,DEF,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,very_high,False
1,Benjamin Sesko,2022-2023,20,Centre-Forward,RB Salzburg,Austrian Bundesliga,24000000,e1290c38854a1ba066c6bbcbe06b189a,2022,ATT,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,high,False
2,Pedri,2022-2023,20,Central Midfield,FC Barcelona,LaLiga,100000000,10e37d554e6c4381fc8e017be60f77e1,2022,MID,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,very_high,False


## 5. Tipos de variables

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   player_name_tm               3 non-null      str     
 1   season                       3 non-null      str     
 2   age                          3 non-null      int64   
 3   position_tm                  3 non-null      str     
 4   club                         3 non-null      str     
 5   league_tm                    3 non-null      str     
 6   market_value_eur             3 non-null      int64   
 7   player_id                    3 non-null      str     
 8   season_start_year            3 non-null      int64   
 9   position_group               3 non-null      str     
 10  log_market_value_eur         3 non-null      float64 
 11  normalized_name              3 non-null      str     
 12  player_name_fbref            3 non-null      str     
 13  squad               

## 6. Valores nulos

In [5]:
df.isna().mean().sort_values(ascending=False).head(20)

player_name_tm          0.0
season                  0.0
age                     0.0
position_tm             0.0
club                    0.0
league_tm               0.0
market_value_eur        0.0
player_id               0.0
season_start_year       0.0
position_group          0.0
log_market_value_eur    0.0
normalized_name         0.0
player_name_fbref       0.0
squad                   0.0
league_fbref            0.0
position_fbref          0.0
minutes_played          0.0
goals_per90             0.0
assists_per90           0.0
shots_per90             0.0
dtype: float64

## 7. Duplicados

In [6]:
df.duplicated(subset=["player_id", "season"]).sum()

np.int64(0)

## 8. Calidad del matching

In [7]:
df["matching_status"].value_counts(dropna=False)

matching_status
matched    3
Name: count, dtype: int64

In [8]:
df["matching_confidence"].describe()

count    3.0
mean     1.0
std      0.0
min      1.0
25%      1.0
50%      1.0
75%      1.0
max      1.0
Name: matching_confidence, dtype: float64

## 9. Distribución del target

In [9]:
df[["market_value_eur", "log_market_value_eur", "minutes_played"]].describe()

,market_value_eur,log_market_value_eur,minutes_played
count,3.000000e+00,3.000000,3.000000
mean,4.966667e+07,17.482877,2326.000000
std,4.359281e+07,0.812418,564.066485
min,2.400000e+07,16.993564,1707.000000
25%,2.450000e+07,17.013975,2083.500000
50%,2.500000e+07,17.034386,2460.000000
75%,6.250000e+07,17.727534,2635.500000
max,1.000000e+08,18.420681,2811.000000


## 10. Distribución por posición

In [10]:
df["position_group"].value_counts(dropna=False)

position_group
DEF    1
ATT    1
MID    1
Name: count, dtype: int64

## 11. Distribución por liga

In [11]:
df["league_tm"].value_counts(dropna=False)

league_tm
Liga Portugal          1
Austrian Bundesliga    1
LaLiga                 1
Name: count, dtype: int64

## 12. Análisis de minutos jugados

In [12]:
df["minutes_played"].describe()

count       3.000000
mean     2326.000000
std       564.066485
min      1707.000000
25%      2083.500000
50%      2460.000000
75%      2635.500000
max      2811.000000
Name: minutes_played, dtype: float64

In [13]:
df["minutes_bucket"].value_counts(dropna=False)

minutes_bucket
very_high    2
high         1
low          0
medium       0
Name: count, dtype: int64

## 13. Revisión de variables de rendimiento

In [14]:
performance_cols = [
    "goals_per90",
    "assists_per90",
    "shots_per90",
    "progressive_passes_per90",
    "progressive_carries_per90",
    "tackles_per90",
    "interceptions_per90",
]

df[performance_cols].describe()

,goals_per90,assists_per90,shots_per90,progressive_passes_per90,progressive_carries_per90,tackles_per90,interceptions_per90
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,0.380000,0.063333,1.590000,4.390000,1.506667,1.226667,1.046667
std,0.400375,0.085049,1.259047,3.118958,0.853249,0.650564,0.758310
min,0.110000,0.000000,0.610000,1.040000,0.730000,0.510000,0.220000
25%,0.150000,0.015000,0.880000,2.980000,1.050000,0.950000,0.715000
50%,0.190000,0.030000,1.150000,4.920000,1.370000,1.390000,1.210000
75%,0.515000,0.095000,2.080000,6.065000,1.895000,1.585000,1.460000
max,0.840000,0.160000,3.010000,7.210000,2.420000,1.780000,1.710000


## 14. Revisión de índices creados

In [15]:
index_cols = [
    "finishing_index",
    "playmaking_index",
    "progression_index",
    "defensive_index",
]

df[index_cols].describe()

,finishing_index,playmaking_index,progression_index,defensive_index
count,3.0,3.0,3.0,3.0
mean,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0


## 15. Conclusiones del análisis exploratorio (EDA v0)

El análisis exploratorio realizado sobre el dataset `player_season_features` permite validar, en primer lugar, la correcta construcción del pipeline de datos y la coherencia estructural del conjunto de datos a nivel jugador-temporada.

### Validación del pipeline e integración de datos

El dataset ha sido generado a partir de la integración de dos fuentes principales: Transfermarkt (valor de mercado y características del jugador) y FBref (métricas de rendimiento). En la muestra analizada, todos los registros presentan un estado de emparejamiento (`matching_status`) correcto y un nivel máximo de confianza (`matching_confidence = 1.0`), lo que confirma que el pipeline de integración funciona adecuadamente en este entorno controlado.

No obstante, es importante destacar que esta muestra es reducida y no representa la complejidad real del problema de *entity resolution*, donde se esperan inconsistencias entre fuentes que deberán ser tratadas en fases posteriores.

---

### Estructura del dataset y calidad de datos

El dataset presenta una estructura consistente, sin valores nulos ni duplicados en las variables clave (`player_id`, `season`). Las variables disponibles incluyen:

* Variables de mercado: `market_value_eur`, `log_market_value_eur`
* Variables de rendimiento por 90 minutos
* Variables derivadas normalizadas (z-score)
* Índices agregados de rendimiento
* Variables de contexto (posición, liga, minutos jugados)

La transformación logarítmica del valor de mercado resulta adecuada para reducir la asimetría de la variable objetivo, lo que facilitará su modelización en fases posteriores.

---

### Distribución del target y variables de contexto

El valor de mercado presenta una elevada dispersión, incluso en esta muestra reducida, lo que refuerza la necesidad de modelarlo en escala logarítmica. Asimismo, los minutos jugados muestran niveles relativamente altos en los jugadores analizados, lo que sugiere que se trata de perfiles consolidados dentro de sus respectivos equipos.

La distribución por posición (`position_group`) y liga (`league_tm`) confirma que el dataset combina jugadores con contextos competitivos heterogéneos, lo que introduce la necesidad de normalizar las métricas de rendimiento para evitar comparaciones sesgadas.

---

### Análisis de métricas de rendimiento

Las variables de rendimiento por 90 minutos presentan variabilidad entre jugadores, especialmente en métricas ofensivas como goles o disparos, lo que sugiere diferencias claras en perfil deportivo.

Sin embargo, estas variables en bruto no son directamente comparables entre jugadores debido a diferencias en posición y liga, lo que justifica la construcción de variables normalizadas.

---

### Validación de las variables derivadas

Los índices de rendimiento (`finishing_index`, `playmaking_index`, `progression_index`, `defensive_index`) se han construido a partir de métricas normalizadas mediante z-score por grupo (posición y liga). En esta muestra inicial, los índices presentan valores constantes (cercanos a cero), lo cual es esperable dado el tamaño reducido de los grupos.

Este comportamiento no representa un problema metodológico, sino una limitación de la muestra. En datasets de mayor tamaño, estos índices permitirán capturar el rendimiento relativo de cada jugador dentro de su contexto competitivo.

---

### Limitaciones del análisis actual

Las principales limitaciones identificadas en esta fase son:

* Tamaño reducido de la muestra, que impide análisis estadísticos robustos
* Ausencia de variabilidad suficiente para evaluar adecuadamente los índices normalizados
* Matching trivial entre fuentes, que no refleja la complejidad real del problema
* Falta de variables adicionales de contexto (por ejemplo, nivel del equipo o competición europea)

Estas limitaciones son inherentes a la fase inicial del proyecto y serán abordadas en etapas posteriores mediante la ampliación del dataset y la mejora del proceso de integración.

---

### Implicaciones para el modelado

A pesar de las limitaciones, el dataset generado constituye una base adecuada para avanzar hacia la fase de modelado, ya que:

* Se dispone de una variable objetivo bien definida (`log_market_value_eur`)
* Se han construido variables explicativas relevantes y normalizadas
* Se ha validado la coherencia estructural del pipeline

El siguiente paso consiste en estimar un modelo baseline que permita evaluar en qué medida las variables de rendimiento explican el valor de mercado observado, y utilizar dicha estimación para identificar posibles ineficiencias en el mercado.

---

### Conclusión

En esta fase, el análisis exploratorio ha cumplido su objetivo principal: validar la calidad, coherencia y estructura del dataset integrado. Aunque no permite aún extraer conclusiones deportivas o econométricas definitivas, sí sienta las bases metodológicas necesarias para avanzar hacia un análisis más profundo con datos reales a mayor escala.